In [0]:
# Import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
# Load the cleaned data
df = spark.table("car_price_cleaned")
df_pd = df.toPandas()

print(f"Original dataset shape: {df_pd.shape}")
print(f"\nColumns: {df_pd.columns.tolist()}")
print(f"\nFirst few rows:")
display(df_pd.head())

Original dataset shape: (205, 28)

Columns: ['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype', 'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg', 'price', 'doornumber_numeric', 'cylindernumber_numeric']

First few rows:


car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,doornumber_numeric,cylindernumber_numeric
1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,2,4
2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,2,4
3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,2,6
4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950.0,4,4
5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450.0,4,5


In [0]:
# Create a copy for feature engineering
df_fe = df_pd.copy()

print("=" * 60)
print("DERIVED FEATURES CREATION")
print("=" * 60)

# 1. Car Age (assuming current year is 2026 and extracting year from CarName if available)
# Since we don't have year column, we'll use symboling as proxy for age estimation
# Lower symboling typically means newer/safer cars
df_fe['car_age_proxy'] = df_fe['symboling'].apply(lambda x: abs(x - 3) + 1)
print("\n✓ Created: car_age_proxy (based on symboling)")

# 2. Power-to-Weight Ratio (Performance metric)
if 'horsepower' in df_fe.columns and 'curbweight' in df_fe.columns:
    df_fe['power_to_weight'] = df_fe['horsepower'] / (df_fe['curbweight'] / 1000)  # HP per ton
    print("✓ Created: power_to_weight (horsepower per ton)")

# 3. Engine Efficiency (HP per liter)
if 'horsepower' in df_fe.columns and 'enginesize' in df_fe.columns:
    df_fe['engine_efficiency'] = df_fe['horsepower'] / df_fe['enginesize']
    print("✓ Created: engine_efficiency (HP per engine size)")

# 4. Fuel Economy Average
if 'citympg' in df_fe.columns and 'highwaympg' in df_fe.columns:
    df_fe['avg_mpg'] = (df_fe['citympg'] + df_fe['highwaympg']) / 2
    print("✓ Created: avg_mpg (average fuel economy)")

# 5. Car Size (Volume approximation)
if all(col in df_fe.columns for col in ['carlength', 'carwidth', 'carheight']):
    df_fe['car_volume'] = df_fe['carlength'] * df_fe['carwidth'] * df_fe['carheight']
    print("✓ Created: car_volume (approximate car volume)")

# 6. Wheelbase to Length Ratio
if 'wheelbase' in df_fe.columns and 'carlength' in df_fe.columns:
    df_fe['wheelbase_ratio'] = df_fe['wheelbase'] / df_fe['carlength']
    print("✓ Created: wheelbase_ratio (stability metric)")

# 7. Bore-Stroke Ratio (Engine design metric)
if 'boreratio' in df_fe.columns and 'stroke' in df_fe.columns:
    df_fe['bore_stroke_ratio'] = df_fe['boreratio'] / df_fe['stroke']
    print("✓ Created: bore_stroke_ratio (engine design)")

# 8. Price per HP (Value metric)
if 'price' in df_fe.columns and 'horsepower' in df_fe.columns:
    df_fe['price_per_hp'] = df_fe['price'] / df_fe['horsepower']
    print("✓ Created: price_per_hp (value metric)")

# 9. Is Luxury (Based on price threshold)
if 'price' in df_fe.columns:
    luxury_threshold = df_fe['price'].quantile(0.75)
    df_fe['is_luxury'] = (df_fe['price'] > luxury_threshold).astype(int)
    print(f"✓ Created: is_luxury (price > ${luxury_threshold:,.2f})")

# 10. Is High Performance (Based on horsepower)
if 'horsepower' in df_fe.columns:
    hp_threshold = df_fe['horsepower'].quantile(0.75)
    df_fe['is_high_performance'] = (df_fe['horsepower'] > hp_threshold).astype(int)
    print(f"✓ Created: is_high_performance (HP > {hp_threshold})")

print(f"\n\nNew dataset shape: {df_fe.shape}")
print(f"New features added: {df_fe.shape[1] - df_pd.shape[1]}")

DERIVED FEATURES CREATION

✓ Created: car_age_proxy (based on symboling)
✓ Created: power_to_weight (horsepower per ton)
✓ Created: engine_efficiency (HP per engine size)
✓ Created: avg_mpg (average fuel economy)
✓ Created: car_volume (approximate car volume)
✓ Created: wheelbase_ratio (stability metric)
✓ Created: bore_stroke_ratio (engine design)
✓ Created: price_per_hp (value metric)
✓ Created: is_luxury (price > $16,503.00)
✓ Created: is_high_performance (HP > 116.0)


New dataset shape: (205, 38)
New features added: 10


In [0]:
# Display new derived features
print("=" * 60)
print("DERIVED FEATURES PREVIEW")
print("=" * 60)

# Get only the newly created features
original_cols = set(df_pd.columns)
new_cols = [col for col in df_fe.columns if col not in original_cols]

print(f"\nNewly Created Features ({len(new_cols)}):")
for col in new_cols:
    print(f"  - {col}")

if new_cols:
    print("\nStatistics for New Features:")
    display(df_fe[new_cols].describe())
    
    print("\nSample values:")
    display(df_fe[new_cols].head(10))

DERIVED FEATURES PREVIEW

Newly Created Features (10):
  - car_age_proxy
  - power_to_weight
  - engine_efficiency
  - avg_mpg
  - car_volume
  - wheelbase_ratio
  - bore_stroke_ratio
  - price_per_hp
  - is_luxury
  - is_high_performance

Statistics for New Features:


car_age_proxy,power_to_weight,engine_efficiency,avg_mpg,car_volume,wheelbase_ratio,bore_stroke_ratio,price_per_hp,is_luxury,is_high_performance
205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0
3.1658536585365855,40.062922773923134,0.8223310643350384,27.985365853658536,618719.2888731707,0.5681705443754281,1.0341666292744793,123.21471146233316,0.24878048780487805,0.24390243902439024
1.2453068281055297,9.570170789259855,0.1829248308216548,6.666037751300955,79463.19526191443,0.02030043767515775,0.14905482197986392,38.972017602715106,0.4333646017486667,0.43048580656147617
1.0,19.935691318327976,0.509090909090909,15.0,452643.156,0.5248815165876777,0.7723342939481268,72.83620689655173,0.0,0.0
2.0,34.18803418803419,0.711340206185567,22.5,566490.6,0.5557422969187675,0.9350282485875706,96.57954545454545,0.0,0.0
3.0,37.80068728522337,0.7798165137614679,27.0,601385.7000000001,0.5651162790697675,0.9835164835164835,110.63636363636364,0.0,0.0
4.0,44.48871181938911,0.8839779005524862,32.0,666250.2000000001,0.5779325120514195,1.0678466076696165,139.6590909090909,0.0,0.0
6.0,85.56149732620321,1.6875,51.5,846007.6590000001,0.6265060240963856,1.5799086757990868,256.9105691056911,1.0,1.0



Sample values:


car_age_proxy,power_to_weight,engine_efficiency,avg_mpg,car_volume,wheelbase_ratio,bore_stroke_ratio,price_per_hp,is_luxury,is_high_performance
1,43.56357927786499,0.8538461538461538,24.0,528019.904,0.5248815165876777,1.294776119402985,121.57657657657657,0,0
1,43.56357927786499,0.8538461538461538,24.0,528019.904,0.5248815165876777,1.294776119402985,148.64864864864865,0,0
3,54.55189514700673,1.013157894736842,22.5,587592.6399999999,0.5519859813084113,0.7723342939481268,107.14285714285714,0,1
2,43.64569961489088,0.9357798165137615,27.0,634816.956,0.565118912797282,0.9382352941176471,136.76470588235293,0,0
2,40.72237960339944,0.8455882352941176,20.0,636734.8319999999,0.5628539071347679,0.9382352941176471,151.7391304347826,1,0
2,43.87714399680893,0.8088235294117647,22.0,624189.969,0.562887760857304,0.9382352941176471,138.63636363636363,0,0
3,38.67791842475387,0.8088235294117647,22.0,766364.0460000001,0.5490399584846912,0.9382352941176471,161.0,1,0
3,37.23764387271496,0.8088235294117647,22.0,766364.0460000001,0.5490399584846912,0.9382352941176471,172.0,1,0
3,45.36616979909268,1.0687022900763359,18.5,769115.802,0.5490399584846912,0.9205882352941176,170.53571428571428,1,1
4,52.40746806419915,1.2213740458015268,19.0,629188.56,0.5583613916947251,0.9205882352941176,111.61979375000001,1,1


In [0]:
# Interaction Features (Product of highly correlated features)
print("=" * 60)
print("INTERACTION FEATURES")
print("=" * 60)

# Create interactions between important numerical features
if all(col in df_fe.columns for col in ['enginesize', 'horsepower']):
    df_fe['engine_power_interaction'] = df_fe['enginesize'] * df_fe['horsepower']
    print("✓ Created: engine_power_interaction")

if all(col in df_fe.columns for col in ['curbweight', 'carlength']):
    df_fe['size_weight_interaction'] = df_fe['curbweight'] * df_fe['carlength']
    print("✓ Created: size_weight_interaction")

if all(col in df_fe.columns for col in ['citympg', 'enginesize']):
    df_fe['efficiency_size_interaction'] = df_fe['citympg'] * df_fe['enginesize']
    print("✓ Created: efficiency_size_interaction")

print(f"\nTotal features now: {df_fe.shape[1]}")

INTERACTION FEATURES
✓ Created: engine_power_interaction
✓ Created: size_weight_interaction
✓ Created: efficiency_size_interaction

Total features now: 41


In [0]:
# Handle text-based numerical columns
print("=" * 60)
print("TEXT TO NUMERIC CONVERSION")
print("=" * 60)

# Convert door numbers from text to numeric
if 'doornumber' in df_fe.columns:
    door_mapping = {'two': 2, 'four': 4}
    df_fe['doornumber_numeric'] = df_fe['doornumber'].map(door_mapping)
    print("✓ Converted: doornumber to doornumber_numeric")
    print(f"   Mapping: {door_mapping}")

# Convert cylinder numbers from text to numeric
if 'cylindernumber' in df_fe.columns:
    cylinder_mapping = {
        'two': 2, 'three': 3, 'four': 4, 'five': 5,
        'six': 6, 'eight': 8, 'twelve': 12
    }
    df_fe['cylindernumber_numeric'] = df_fe['cylindernumber'].map(cylinder_mapping)
    print("✓ Converted: cylindernumber to cylindernumber_numeric")
    print(f"   Mapping: {cylinder_mapping}")

print("\nText-to-numeric conversion complete!")

TEXT TO NUMERIC CONVERSION
✓ Converted: doornumber to doornumber_numeric
   Mapping: {'two': 2, 'four': 4}
✓ Converted: cylindernumber to cylindernumber_numeric
   Mapping: {'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'eight': 8, 'twelve': 12}

Text-to-numeric conversion complete!


In [0]:
# Label Encoding for Ordinal Categorical Variables
print("=" * 60)
print("LABEL ENCODING (ORDINAL VARIABLES)")
print("=" * 60)

# Create a copy of original categorical columns for reference
df_fe_encoded = df_fe.copy()

# Identify categorical columns
categorical_cols = df_fe.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns found: {len(categorical_cols)}")
print(categorical_cols)

# Apply Label Encoding
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_fe_encoded[f'{col}_encoded'] = le.fit_transform(df_fe[col])
    label_encoders[col] = le
    
    print(f"\n✓ Encoded: {col}")
    print(f"   Classes: {list(le.classes_)}")
    print(f"   Encoded as: {list(range(len(le.classes_)))}")

print(f"\n\nTotal encoded features created: {len(categorical_cols)}")

LABEL ENCODING (ORDINAL VARIABLES)

Categorical columns found: 10
['CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']

✓ Encoded: CarName
   Classes: ['Nissan versa', 'alfa-romero Quadrifoglio', 'alfa-romero giulia', 'alfa-romero stelvio', 'audi 100 ls', 'audi 100ls', 'audi 4000', 'audi 5000', 'audi 5000s (diesel)', 'audi fox', 'bmw 320i', 'bmw x1', 'bmw x3', 'bmw x4', 'bmw x5', 'bmw z4', 'buick century', 'buick century luxus (sw)', 'buick century special', 'buick electra 225 custom', 'buick opel isuzu deluxe', 'buick regal sport coupe (turbo)', 'buick skyhawk', 'buick skylark', 'chevrolet impala', 'chevrolet monte carlo', 'chevrolet vega 2300', 'dodge challenger se', 'dodge colt (sw)', 'dodge colt hardtop', 'dodge coronet custom', 'dodge coronet custom (sw)', 'dodge d200', 'dodge dart custom', 'dodge monaco (sw)', 'dodge rampage', 'honda accord', 'honda accord cvcc', 'honda accord lx', 'honda civi

In [0]:
# One-Hot Encoding for Nominal Categorical Variables
print("=" * 60)
print("ONE-HOT ENCODING (NOMINAL VARIABLES)")
print("=" * 60)

# Select categorical columns for one-hot encoding
# We'll use columns with fewer unique values to avoid creating too many features
categorical_cols = df_fe.select_dtypes(include=['object']).columns.tolist()

print("\nApplying One-Hot Encoding...\n")

# Apply one-hot encoding
df_fe_onehot = pd.get_dummies(df_fe, columns=categorical_cols, prefix=categorical_cols, drop_first=True)

print(f"Original features: {df_fe.shape[1]}")
print(f"After one-hot encoding: {df_fe_onehot.shape[1]}")
print(f"New dummy features created: {df_fe_onehot.shape[1] - df_fe.shape[1]}")

# Show which dummy features were created
new_dummy_cols = [col for col in df_fe_onehot.columns if col not in df_fe.columns]
print(f"\nDummy columns created ({len(new_dummy_cols)}):")
for i, col in enumerate(new_dummy_cols[:20], 1):  # Show first 20
    print(f"  {i}. {col}")
if len(new_dummy_cols) > 20:
    print(f"  ... and {len(new_dummy_cols) - 20} more")

print("\n✓ One-hot encoding complete!")

ONE-HOT ENCODING (NOMINAL VARIABLES)

Applying One-Hot Encoding...

Original features: 41
After one-hot encoding: 206
New dummy features created: 165

Dummy columns created (175):
  1. CarName_alfa-romero Quadrifoglio
  2. CarName_alfa-romero giulia
  3. CarName_alfa-romero stelvio
  4. CarName_audi 100 ls
  5. CarName_audi 100ls
  6. CarName_audi 4000
  7. CarName_audi 5000
  8. CarName_audi 5000s (diesel)
  9. CarName_audi fox
  10. CarName_bmw 320i
  11. CarName_bmw x1
  12. CarName_bmw x3
  13. CarName_bmw x4
  14. CarName_bmw x5
  15. CarName_bmw z4
  16. CarName_buick century
  17. CarName_buick century luxus (sw)
  18. CarName_buick century special
  19. CarName_buick electra 225 custom
  20. CarName_buick opel isuzu deluxe
  ... and 155 more

✓ One-hot encoding complete!


In [0]:
# Feature Scaling
print("=" * 60)
print("FEATURE SCALING")
print("=" * 60)

# Create a copy for scaling
df_scaled = df_fe_onehot.copy()

# Select numerical columns for scaling (exclude target and ID columns)
numerical_cols = df_scaled.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [col for col in numerical_cols if 'ID' not in col and 'id' not in col and col != 'price']

print(f"\nNumerical columns to scale: {len(numerical_cols)}\n")

# StandardScaler (mean=0, std=1)
scaler_standard = StandardScaler()
df_scaled_standard = df_scaled.copy()
df_scaled_standard[numerical_cols] = scaler_standard.fit_transform(df_scaled[numerical_cols])

print("✓ StandardScaler applied (mean=0, std=1)")
print(f"   Features scaled: {len(numerical_cols)}")

# MinMaxScaler (range 0-1)
scaler_minmax = MinMaxScaler()
df_scaled_minmax = df_scaled.copy()
df_scaled_minmax[numerical_cols] = scaler_minmax.fit_transform(df_scaled[numerical_cols])

print("✓ MinMaxScaler applied (range 0-1)")
print(f"   Features scaled: {len(numerical_cols)}")

print("\nScaling complete! Two versions available:")
print("  1. df_scaled_standard (StandardScaler)")
print("  2. df_scaled_minmax (MinMaxScaler)")

FEATURE SCALING

Numerical columns to scale: 28

✓ StandardScaler applied (mean=0, std=1)
   Features scaled: 28
✓ MinMaxScaler applied (range 0-1)
   Features scaled: 28

Scaling complete! Two versions available:
  1. df_scaled_standard (StandardScaler)
  2. df_scaled_minmax (MinMaxScaler)


In [0]:
# Feature Selection - Select K Best
print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)

# Prepare data for feature selection
if 'price' in df_scaled_minmax.columns:
    X = df_scaled_minmax.drop('price', axis=1)
    y = df_scaled_minmax['price']
    
    # Remove any remaining non-numeric columns
    X_numeric = X.select_dtypes(include=['int64', 'float64'])
    
    print(f"\nTotal features available: {X_numeric.shape[1]}")
    print(f"Target variable: price")
    
    # Select top K features using f_regression
    k = min(20, X_numeric.shape[1])  # Select top 20 or all if fewer
    selector = SelectKBest(score_func=f_regression, k=k)
    X_selected = selector.fit_transform(X_numeric, y)
    
    # Get selected feature names
    selected_features = X_numeric.columns[selector.get_support()].tolist()
    feature_scores = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Score': selector.scores_
    }).sort_values('Score', ascending=False)
    
    print(f"\n✓ Selected top {k} features using f_regression")
    print("\nTop 20 Features by Score:")
    display(feature_scores.head(20))
    
    print(f"\nSelected features for modeling:")
    for i, feat in enumerate(selected_features, 1):
        print(f"  {i}. {feat}")
else:
    print("\nPrice column not found. Skipping feature selection.")

FEATURE SELECTION

Total features available: 30
Target variable: price

✓ Selected top 20 features using f_regression

Top 20 Features by Score:


Feature,Score
enginesize,657.6404214943283
engine_power_interaction,604.233806748267
curbweight,468.5944305774156
size_weight_interaction,428.80654062458336
horsepower,382.1634091788086
is_luxury,356.8800265663244
price_per_hp,294.7731490710355
carwidth,276.42364586085665
is_high_performance,230.58517163331877
cylindernumber_numeric,216.38850169045273



Selected features for modeling:
  1. wheelbase
  2. carlength
  3. carwidth
  4. curbweight
  5. enginesize
  6. boreratio
  7. horsepower
  8. citympg
  9. highwaympg
  10. cylindernumber_numeric
  11. power_to_weight
  12. avg_mpg
  13. car_volume
  14. wheelbase_ratio
  15. price_per_hp
  16. is_luxury
  17. is_high_performance
  18. engine_power_interaction
  19. size_weight_interaction
  20. efficiency_size_interaction


In [0]:
# Summary of Feature Engineering
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print(f"\n1. ORIGINAL DATASET:")
print(f"   - Rows: {df_pd.shape[0]}")
print(f"   - Columns: {df_pd.shape[1]}")

print(f"\n2. DERIVED FEATURES:")
original_cols = set(df_pd.columns)
derived_cols = [col for col in df_fe.columns if col not in original_cols]
print(f"   - New features created: {len(derived_cols)}")
print(f"   - Total columns: {df_fe.shape[1]}")

print(f"\n3. AFTER ONE-HOT ENCODING:")
print(f"   - Total columns: {df_fe_onehot.shape[1]}")
print(f"   - Dummy variables: {df_fe_onehot.shape[1] - df_fe.shape[1]}")

print(f"\n4. ENCODING METHODS APPLIED:")
print(f"   ✓ Label Encoding (ordinal variables)")
print(f"   ✓ One-Hot Encoding (nominal variables)")
print(f"   ✓ StandardScaler (normalization)")
print(f"   ✓ MinMaxScaler (range scaling)")

print(f"\n5. FEATURE TYPES:")
print(f"   - Original features: {df_pd.shape[1]}")
print(f"   - Derived features: {len(derived_cols)}")
print(f"   - Interaction features: {df_fe.shape[1] - df_pd.shape[1] - len(derived_cols)}")
print(f"   - After encoding: {df_fe_onehot.shape[1]}")

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE!")
print("=" * 60)

FEATURE ENGINEERING SUMMARY

1. ORIGINAL DATASET:
   - Rows: 205
   - Columns: 28

2. DERIVED FEATURES:
   - New features created: 13
   - Total columns: 41

3. AFTER ONE-HOT ENCODING:
   - Total columns: 206
   - Dummy variables: 165

4. ENCODING METHODS APPLIED:
   ✓ Label Encoding (ordinal variables)
   ✓ One-Hot Encoding (nominal variables)
   ✓ StandardScaler (normalization)
   ✓ MinMaxScaler (range scaling)

5. FEATURE TYPES:
   - Original features: 28
   - Derived features: 13
   - Interaction features: 0
   - After encoding: 206

FEATURE ENGINEERING COMPLETE!


In [0]:
# Save Engineered Features
print("="*60)
print("SAVING ENGINEERED FEATURES")
print("="*60)

# Save different versions
print("\nSaving multiple versions of the dataset...\n")

# 1. With derived features only (no encoding)
df_derived_spark = spark.createDataFrame(df_fe)
df_derived_spark.write.mode("overwrite").saveAsTable("car_price_features_derived")
print(f"✓ Saved: car_price_features_derived")
print(f"   Shape: {df_fe.shape}")
print(f"   Description: Original + derived features")

# 2. With label encoding
df_encoded_spark = spark.createDataFrame(df_fe_encoded)
df_encoded_spark.write.mode("overwrite").saveAsTable("car_price_features_labeled")
print(f"\n✓ Saved: car_price_features_labeled")
print(f"   Shape: {df_fe_encoded.shape}")
print(f"   Description: With label encoding")

# 3. With one-hot encoding (sanitize column names and convert uint8 to int32)
import re
df_fe_onehot_fixed = df_fe_onehot.copy()
# Convert uint8 to int32
for col in df_fe_onehot_fixed.columns:
    if df_fe_onehot_fixed[col].dtype == 'uint8':
        df_fe_onehot_fixed[col] = df_fe_onehot_fixed[col].astype('int32')
# Sanitize column names
df_fe_onehot_fixed.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in df_fe_onehot_fixed.columns]

df_onehot_spark = spark.createDataFrame(df_fe_onehot_fixed)
df_onehot_spark.write.mode("overwrite").saveAsTable("car_price_features_onehot")
print(f"\n✓ Saved: car_price_features_onehot")
print(f"   Shape: {df_fe_onehot.shape}")
print(f"   Description: With one-hot encoding (ready for ML)")

# 4. Scaled version (MinMax) - also sanitize and convert uint8 to int32
df_scaled_minmax_fixed = df_scaled_minmax.copy()
# Convert uint8 to int32
for col in df_scaled_minmax_fixed.columns:
    if df_scaled_minmax_fixed[col].dtype == 'uint8':
        df_scaled_minmax_fixed[col] = df_scaled_minmax_fixed[col].astype('int32')
# Sanitize column names
df_scaled_minmax_fixed.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in df_scaled_minmax_fixed.columns]

df_scaled_spark = spark.createDataFrame(df_scaled_minmax_fixed)
df_scaled_spark.write.mode("overwrite").saveAsTable("car_price_features_scaled")
print(f"\n✓ Saved: car_price_features_scaled")
print(f"   Shape: {df_scaled_minmax.shape}")
print(f"   Description: One-hot encoded + MinMax scaled")

print("\n" + "="*60)
print("ALL DATASETS SAVED SUCCESSFULLY!")
print("="*60)

print("\nAvailable tables:")
print("  1. car_price_features_derived   - For EDA and manual modeling")
print("  2. car_price_features_labeled   - For tree-based models")
print("  3. car_price_features_onehot    - For linear models (unscaled)")
print("  4. car_price_features_scaled    - For models requiring scaled features")

SAVING ENGINEERED FEATURES

Saving multiple versions of the dataset...

✓ Saved: car_price_features_derived
   Shape: (205, 41)
   Description: Original + derived features

✓ Saved: car_price_features_labeled
   Shape: (205, 51)
   Description: With label encoding

✓ Saved: car_price_features_onehot
   Shape: (205, 206)
   Description: With one-hot encoding (ready for ML)

✓ Saved: car_price_features_scaled
   Shape: (205, 206)
   Description: One-hot encoded + MinMax scaled

ALL DATASETS SAVED SUCCESSFULLY!

Available tables:
  1. car_price_features_derived   - For EDA and manual modeling
  2. car_price_features_labeled   - For tree-based models
  3. car_price_features_onehot    - For linear models (unscaled)
  4. car_price_features_scaled    - For models requiring scaled features


In [0]:
# Save Engineered Features to Workspace for Download
print("="*60)
print("SAVING TO WORKSPACE FOR DOWNLOAD")
print("="*60)

print("\nLoading engineered datasets from tables...\n")

# Load all engineered datasets from Spark tables
df_derived = spark.table("car_price_features_derived").toPandas()
df_labeled = spark.table("car_price_features_labeled").toPandas()
df_onehot = spark.table("car_price_features_onehot").toPandas()
df_scaled = spark.table("car_price_features_scaled").toPandas()

print("✓ Loaded all 4 datasets from tables")
print("\nSaving as CSV files to workspace...\n")

# Use workspace directory (works on serverless compute)
workspace_path = "/Workspace/Users/atukundashakira1@gmail.com/car_price_datasets/"

import os
os.makedirs(workspace_path, exist_ok=True)

# 1. Save derived features
file1 = f"{workspace_path}car_price_features_derived.csv"
df_derived.to_csv(file1, index=False)
print(f"✓ Saved: car_price_features_derived.csv")
print(f"   Rows: {df_derived.shape[0]:,} | Columns: {df_derived.shape[1]}")
print(f"   Description: Original + derived features (best for EDA)")

# 2. Save label encoded version
file2 = f"{workspace_path}car_price_features_labeled.csv"
df_labeled.to_csv(file2, index=False)
print(f"\n✓ Saved: car_price_features_labeled.csv")
print(f"   Rows: {df_labeled.shape[0]:,} | Columns: {df_labeled.shape[1]}")
print(f"   Description: With label encoding (for tree-based models)")

# 3. Save one-hot encoded version
file3 = f"{workspace_path}car_price_features_onehot.csv"
df_onehot.to_csv(file3, index=False)
print(f"\n✓ Saved: car_price_features_onehot.csv")
print(f"   Rows: {df_onehot.shape[0]:,} | Columns: {df_onehot.shape[1]}")
print(f"   Description: With one-hot encoding (for linear models)")

# 4. Save scaled version
file4 = f"{workspace_path}car_price_features_scaled.csv"
df_scaled.to_csv(file4, index=False)
print(f"\n✓ Saved: car_price_features_scaled.csv")
print(f"   Rows: {df_scaled.shape[0]:,} | Columns: {df_scaled.shape[1]}")
print(f"   Description: One-hot + MinMax scaled (0-1 range)")

# Calculate file sizes
import glob
files = glob.glob(f"{workspace_path}*.csv")
total_size = sum(os.path.getsize(f) for f in files) / (1024 * 1024)  # MB

print("\n" + "="*60)
print("FILES SAVED SUCCESSFULLY!")
print("="*60)

print(f"\n📁 Location: {workspace_path}")
print(f"💾 Total size: {total_size:.2f} MB")
print(f"📊 Files created: {len(files)}")

print("\n📥 HOW TO DOWNLOAD:\n")
print("Method 1: Via Workspace UI")
print("   1. Click 'Workspace' in the left sidebar")
print("   2. Navigate to: Users → atukundashakira1@gmail.com → car_price_datasets/")
print("   3. Right-click any CSV file → Download")

print("\nMethod 2: Directly from File Browser")
print("   1. Open Databricks file browser")
print("   2. Browse to: /Workspace/Users/atukundashakira1@gmail.com/car_price_datasets/")
print("   3. Select files to download")

print("\n📝 FILES AVAILABLE:\n")
for i, file in enumerate(sorted(files), 1):
    filename = os.path.basename(file)
    size = os.path.getsize(file) / 1024  # KB
    print(f"   {i}. {filename} ({size:.1f} KB)")

print("\n✓ All engineered datasets ready for download!")

SAVING TO WORKSPACE FOR DOWNLOAD

Loading engineered datasets from tables...

✓ Loaded all 4 datasets from tables

Saving as CSV files to workspace...

✓ Saved: car_price_features_derived.csv
   Rows: 205 | Columns: 41
   Description: Original + derived features (best for EDA)

✓ Saved: car_price_features_labeled.csv
   Rows: 205 | Columns: 51
   Description: With label encoding (for tree-based models)

✓ Saved: car_price_features_onehot.csv
   Rows: 205 | Columns: 206
   Description: With one-hot encoding (for linear models)

✓ Saved: car_price_features_scaled.csv
   Rows: 205 | Columns: 206
   Description: One-hot + MinMax scaled (0-1 range)

FILES SAVED SUCCESSFULLY!

📁 Location: /Workspace/Users/atukundashakira1@gmail.com/car_price_datasets/
💾 Total size: 0.39 MB
📊 Files created: 4

📥 HOW TO DOWNLOAD:

Method 1: Via Workspace UI
   1. Click 'Workspace' in the left sidebar
   2. Navigate to: Users → atukundashakira1@gmail.com → car_price_datasets/
   3. Right-click any CSV file → Do